In this file we classify each photo of a particular species with either "Invasive", "Native" or "Introduced non-invasive".

To do so, we take the coordinate of each picture and see in which bounding box it falls into (file "Updated_Complete_Region_Dataset.csv"). If it falls into more than one bounding box, we check for which one it is closer to center.

In [8]:
import pandas as pd
import ast
import math
import os

In [9]:
#change to desired path

PLANT_DISTRIBUTION = './support_files/plant_distribution.csv' #this contains the region where a plant is introduced or native
REGION_COORDINATES_DATASET = './support_files/Updated_Complete_Region_Dataset.csv' #this contains the coordinates for the different regions

#manually identified invasive species
INVASIVE_SPECIES = ['lythrum hyssopifolia', 'lythrum salicaria', 'lythrum virgatum']

In [10]:
#import the dataset into a Pandas DataFrame. Avoid doing it every time the function "generate_classification()" is called.

plant_distribution_df = pd.read_csv(PLANT_DISTRIBUTION)
regions_df = pd.read_csv(REGION_COORDINATES_DATASET)

cols = ['introduced', 'native']

#Since when importing the dataframe into pandas lists are evaluated as string, they need to be reverted to their original form
for col in cols:
    plant_distribution_df[col] = plant_distribution_df[col].apply(lambda x : ast.literal_eval(x) if pd.notnull(x) else x)

regions_df['bbox'] = regions_df['bbox'].apply(lambda x : ast.literal_eval(x) if pd.notnull(x) else x)

In [17]:
def generate_classification(lat: float, lon: float, species_name: str, regions_df: pd.DataFrame, plant_distribution_df: pd.DataFrame) -> str:

    """Given the latitude and longitude of a picture and the name of the species (lowercase separated by a ' ',
    e.g. 'lythrum alatum'), return the classification ('native', 'invasive', 'introduced non-invasive')
    for the picture of the species."""

    current_region_index = -1 #indices can never be -1

    for i, row in regions_df.iterrows():

        if row['bbox'][0]<lat<row['bbox'][1] and row['bbox'][2]<lon<row['bbox'][3]: 
                
            distance = math.dist((lat, lon), (row['lat'], row['lon']))

            if current_region_index == -1:
                current_region_index = i
            else:
                if distance < math.dist((lat, lon), (regions_df.at[current_region_index, 'lat'], regions_df.at[current_region_index, 'lon'])):
                    #update the region that best matches the coordinates
                    current_region_index = i

    if current_region_index == -1:
        print("Failed to locate the coordinates into a region")
        return None

    #Now we have the name of the region to be looked at in the "plant_distribution.csv" file
    region_name = regions_df.at[current_region_index, 'Original name']

    if region_name in plant_distribution_df.loc[plant_distribution_df['Species']==species_name, 'introduced'].iloc[0]:

        if species_name in INVASIVE_SPECIES:
            classification = 'invasive'
        else:
            classification = 'introduced non-invasive'

    elif region_name in plant_distribution_df.loc[plant_distribution_df['Species']==species_name, 'native'].iloc[0]:
        classification = 'native'

    else:
        print(f"Unable to classify this picture for species {species_name}, the region is not native or introduced")
        return None



    return classification
            

In [21]:
folder_path="taxas/metadata"

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        specie=filename.split("_")[0]
        name=filename.split("_")[1]
        complete_name=f"{specie} {name}"
        print(complete_name)
        input_path = os.path.join(folder_path, filename)
        output_path = os.path.join(folder_path, filename)

        df = pd.read_csv(input_path)

        df["label"] = df.apply(
            lambda row: generate_classification(row["latitude"], row["longitude"], complete_name, regions_df, plant_distribution_df),
            axis=1
        )

        df.to_csv(output_path, index=False)
        print(f"Updates CSV to {output_path}")


lythrum acutangulum
Updates CSV to taxas/metadata\lythrum_acutangulum_metadata.csv
lythrum alatum
Failed to locate the coordinates into a region
Failed to locate the coordinates into a region
Failed to locate the coordinates into a region
Unable to classify this picture for species lythrum alatum, the region is not native or introduced
Unable to classify this picture for species lythrum alatum, the region is not native or introduced
Unable to classify this picture for species lythrum alatum, the region is not native or introduced
Unable to classify this picture for species lythrum alatum, the region is not native or introduced
Failed to locate the coordinates into a region
Failed to locate the coordinates into a region
Failed to locate the coordinates into a region
Unable to classify this picture for species lythrum alatum, the region is not native or introduced
Unable to classify this picture for species lythrum alatum, the region is not native or introduced
Failed to locate the coord

In [20]:
def count_missing_values(df):
    missing_label = df['label'].isna().sum()
    missing_latitude = df['latitude'].isna().sum()
    missing_longitude = df['longitude'].isna().sum()
    
    print(f"Missing 'label': {missing_label}")
    print(f"Missing 'latitude': {missing_latitude}")
    print(f"Missing 'longitude': {missing_longitude}")

count_missing_values(pd.read_csv("taxas\metadata\lythrum_alatum_metadata.csv"))

Missing 'label': 125
Missing 'latitude': 86
Missing 'longitude': 86


<>:10: SyntaxWarning: invalid escape sequence '\m'
<>:10: SyntaxWarning: invalid escape sequence '\m'
C:\Users\babac\AppData\Local\Temp\ipykernel_23592\2325552265.py:10: SyntaxWarning: invalid escape sequence '\m'
  count_missing_values(pd.read_csv("taxas\metadata\lythrum_alatum_metadata.csv"))
